In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

In [2]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()

In [3]:
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

In [4]:
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, :-1], df.iloc[:, -1], test_size=0.25, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

X_train = torch.FloatTensor(X_train.values)
X_val = torch.FloatTensor(X_val.values)
X_test = torch.FloatTensor(X_test.values)
means = X_train.mean(dim=0, keepdim=True)
stds = X_train.std(dim=0, keepdim=True)
X_train = (X_train - means) / stds
X_val = (X_val - means) / stds
X_test = (X_test - means) / stds

y_train = torch.FloatTensor(y_train.values).reshape(-1, 1)
y_val = torch.FloatTensor(y_val.values).reshape(-1, 1)
y_test = torch.FloatTensor(y_test.values).reshape(-1, 1)

In [6]:
torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [7]:
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

Epoch 1/20, Loss: 16.074838638305664
Epoch 2/20, Loss: 4.7494049072265625
Epoch 3/20, Loss: 2.182666540145874
Epoch 4/20, Loss: 1.29657781124115
Epoch 5/20, Loss: 0.9541639089584351
Epoch 6/20, Loss: 0.8101152181625366
Epoch 7/20, Loss: 0.7419664859771729
Epoch 8/20, Loss: 0.7042079567909241
Epoch 9/20, Loss: 0.6794491410255432
Epoch 10/20, Loss: 0.6608676314353943
Epoch 11/20, Loss: 0.6456913948059082
Epoch 12/20, Loss: 0.63272625207901
Epoch 13/20, Loss: 0.6214048266410828
Epoch 14/20, Loss: 0.6114156246185303
Epoch 15/20, Loss: 0.602558434009552
Epoch 16/20, Loss: 0.5946851372718811
Epoch 17/20, Loss: 0.5876765847206116
Epoch 18/20, Loss: 0.581432044506073
Epoch 19/20, Loss: 0.5758640170097351
Epoch 20/20, Loss: 0.5708959102630615


In [8]:
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)

In [9]:
for param in model.named_parameters():
    print(param)

('weight', Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True))
('bias', Parameter containing:
tensor([0.3117], requires_grad=True))


In [10]:
model(X_train[:2])

tensor([[0.9811],
        [0.1410]], grad_fn=<AddmmBackward0>)

In [11]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [12]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [13]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.313408851623535
Epoch 2/20, Loss: 0.7835941910743713
Epoch 3/20, Loss: 0.6289426684379578
Epoch 4/20, Loss: 0.6097182631492615
Epoch 5/20, Loss: 0.5992293953895569
Epoch 6/20, Loss: 0.5908074378967285
Epoch 7/20, Loss: 0.5835899114608765
Epoch 8/20, Loss: 0.5772803425788879
Epoch 9/20, Loss: 0.5717209577560425
Epoch 10/20, Loss: 0.5668057203292847
Epoch 11/20, Loss: 0.5624527931213379
Epoch 12/20, Loss: 0.5585941672325134
Epoch 13/20, Loss: 0.5551713705062866
Epoch 14/20, Loss: 0.5521337389945984
Epoch 15/20, Loss: 0.5494365692138672
Epoch 16/20, Loss: 0.5470404624938965
Epoch 17/20, Loss: 0.5449107885360718
Epoch 18/20, Loss: 0.5430170893669128
Epoch 19/20, Loss: 0.5413322448730469
Epoch 20/20, Loss: 0.539832592010498


In [14]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50,40),
    nn.ReLU(),
    nn.Linear(40,1)
)

learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

for i in range(n_epochs):
    model.zero_grad()
    y_pred = model(X_train)
    loss = mse(y_pred, y_train)
    loss.backward()
    optimizer.step()
    print(f"{i}:step, loss:{loss}")

0:step, loss:5.031027793884277
1:step, loss:2.0860512256622314
2:step, loss:1.0304265022277832
3:step, loss:0.8857324123382568
4:step, loss:0.8032048344612122
5:step, loss:0.7519382238388062
6:step, loss:0.7190009951591492
7:step, loss:0.6970219612121582
8:step, loss:0.681583046913147
9:step, loss:0.6700412034988403
10:step, loss:0.6607772707939148
11:step, loss:0.6528614163398743
12:step, loss:0.6458008885383606
13:step, loss:0.639346182346344
14:step, loss:0.6332712769508362
15:step, loss:0.6274890899658203
16:step, loss:0.6219675540924072
17:step, loss:0.6166506409645081
18:step, loss:0.6115112900733948
19:step, loss:0.6065381169319153


In [15]:
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
device = "cuda"
learning_rate = 0.0001

model = model.to(device)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True, persistent_workers=True, num_workers=8)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            X_batch, y_batch = X_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
        
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 0.5785
Epoch 2/20, Loss: 0.5386
Epoch 3/20, Loss: 0.5078
Epoch 4/20, Loss: 0.4853
Epoch 5/20, Loss: 0.4666
Epoch 6/20, Loss: 0.4519
Epoch 7/20, Loss: 0.4396
Epoch 8/20, Loss: 0.4287
Epoch 9/20, Loss: 0.4193
Epoch 10/20, Loss: 0.4118
Epoch 11/20, Loss: 0.4050
Epoch 12/20, Loss: 0.3994
Epoch 13/20, Loss: 0.3941
Epoch 14/20, Loss: 0.3903
Epoch 15/20, Loss: 0.3864
Epoch 16/20, Loss: 0.3831
Epoch 17/20, Loss: 0.3799
Epoch 18/20, Loss: 0.3770
Epoch 19/20, Loss: 0.3743
Epoch 20/20, Loss: 0.3717


In [16]:
def evaluate(model, data_loader, metric_fn, aggregate_fn = torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
        return aggregate_fn(torch.stack(metrics))

In [17]:
valid_dataset = TensorDataset(X_val, y_val)
valid_loader = DataLoader(valid_dataset, batch_size=32)
valid_mse = evaluate(model, valid_loader, mse)
valid_mse

tensor(0.3477, device='cuda:0')

In [18]:
def rmse(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean().sqrt()

valid_mse = evaluate(model, valid_loader, rmse, aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics)))
valid_mse

tensor(0.7591, device='cuda:0')

In [19]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
        return metric.compute()

In [20]:
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
evaluate_tm(model, valid_loader, rmse)

tensor(0.5902, device='cuda:0')

In [ ]:
uv pip install torchvision --reinstall --index-url https://download.pytorch.org/whl/cu130
